# Exploratory Data Analysis: IMDb Movie Reviews Sentiment Analysis

This notebook explores the official IMDb 50k Movie Review dataset used for binary sentiment classification.

### Objectives:
1. Inspect dataset dimensions, schema, and sample reviews.
2. Check for missing values and duplicates.
3. Analyze class distribution (balance vs. imbalance).
4. Analyze review length (character and word distributions) across sentiment classes.
5. Examine characteristic vocabulary and key sentiment terms.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data_loader import load_processed_splits, LABEL_MAPPING
from src.preprocessing import clean_html, normalize_whitespace

sns.set_theme(style="whitegrid")
print("Libraries successfully imported.")

## 1. Load Processed Partitions

In [ ]:
train_df, val_df, test_df = load_processed_splits(str(project_root / "data" / "processed"))
print(f"Train samples: {len(train_df):,}")
print(f"Val samples:   {len(val_df):,}")
print(f"Test samples:  {len(test_df):,}")
train_df.head(5)

## 2. Check Missing Values & Duplicates

In [ ]:
print("Missing values in Train:")
print(train_df.isnull().sum())
duplicates = train_df.duplicated(subset=["review"]).sum()
print(f"\nDuplicate reviews in Train partition: {duplicates:,}")

## 3. Class Distribution Analysis

In [ ]:
class_counts = train_df["label"].value_counts().rename(index=LABEL_MAPPING)
print("Class distribution:")
print(class_counts)

plt.figure(figsize=(6, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, palette=["#d9534f", "#5cb85c"])
plt.title("Class Distribution (Training Split)")
plt.ylabel("Number of Reviews")
plt.show()

## 4. Review Length & Word Count Distributions

In [ ]:
train_df["clean_text"] = train_df["review"].apply(lambda t: normalize_whitespace(clean_html(str(t))))
train_df["word_count"] = train_df["clean_text"].apply(lambda t: len(t.split()))
train_df["sentiment"] = train_df["label"].map(LABEL_MAPPING)

print("Word count descriptive statistics:")
print(train_df.groupby("sentiment")["word_count"].describe())

plt.figure(figsize=(10, 5))
sns.histplot(data=train_df, x="word_count", hue="sentiment", kde=True, bins=50, palette=["#d9534f", "#5cb85c"])
plt.title("Review Word Count Distribution by Sentiment")
plt.xlabel("Words per Review")
plt.xlim(0, 1000)
plt.show()

## 5. Key Findings and Dataset Quality Observations
- **Class Balance**: The IMDb dataset is well balanced (~50% negative, ~50% positive). Aggressive synthetic oversampling (SMOTE) or extreme weighting is unnecessary, though standard stratified sampling is maintained to safeguard against skew.
- **Text Length**: Most reviews contain between 100 and 400 words. A maximum sequence length of 256 for DistilBERT covers the vast majority of core sentiment signals while maintaining reasonable memory footprint.
- **HTML Artifacts**: Raw reviews contain occasional `<br /><br />` tags, which are cleanly stripped during preprocessing.